In [5]:
import requests
from geopy.distance import geodesic

# 사용자 위치 (예: 가산디지털단지)
user_lat = 37.4812
user_lng = 126.8828

# 카카오 REST API 키
KAKAO_API_KEY = "b3e4c845d73f26ab456c89a38561b720"

# 요청 설정
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
params = {
    "query": "롯데시네마",
    "x": user_lng,
    "y": user_lat,
    "radius": 20000,
    "size": 15  # 최대한 많이 받아오기
}

# API 요청
res = requests.get("https://dapi.kakao.com/v2/local/search/keyword.json", params=params, headers=headers)
places = res.json().get("documents", [])

# 거리 기준으로 가장 가까운 지점 선택
closest = None
min_dist = float('inf')

for place in places:
    name = place["place_name"]
    lat = float(place["y"])
    lng = float(place["x"])
    distance = geodesic((user_lat, user_lng), (lat, lng)).km

    if "롯데시네마" in name and distance < min_dist:
        min_dist = distance
        closest = {
            "name": name,
            "lat": lat,
            "lng": lng,
            "distance": round(distance, 2),
            "address": place.get("road_address_name", ""),
        }

# 결과 출력
if closest:
    print(f"📍 가장 가까운 롯데시네마: {closest['name']}")
    print(f"📏 거리: {closest['distance']} km")
    print(f"🏠 주소: {closest['address']}")
else:
    print("❌ 롯데시네마 검색 실패 또는 근처에 없음.")


📍 가장 가까운 롯데시네마: 롯데시네마 가산디지털
📏 거리: 0.66 km
🏠 주소: 서울 금천구 디지털로10길 9


In [1]:
import requests
from geopy.distance import geodesic
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

# 카카오맵 API 키 (본인 키로 교체)
KAKAO_API_KEY = "b3e4c845d73f26ab456c89a38561b720"

# cinemaID 매핑 (사용자 직접 관리)
cinema_map = {
    "가산": ("1013", "1"),
    "건대": ("1004", "1"),
    "동탄": ("3048", "2"),
    "서청주": ("4004", "3"),
    # 필요 시 추가
}

def find_nearest_lottecinema(user_lat, user_lng):
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {
        "query": "롯데시네마",
        "x": user_lng,
        "y": user_lat,
        "radius": 20000,
        "size": 15
    }
    res = requests.get("https://dapi.kakao.com/v2/local/search/keyword.json", params=params, headers=headers)
    places = res.json().get("documents", [])

    closest = None
    min_dist = float('inf')

    for place in places:
        name = place["place_name"]
        lat = float(place["y"])
        lng = float(place["x"])
        distance = geodesic((user_lat, user_lng), (lat, lng)).km

        if "롯데시네마" in name and distance < min_dist:
            min_dist = distance
            closest = {
                "name": name,
                "lat": lat,
                "lng": lng,
                "distance": round(distance, 2),
                "address": place.get("road_address_name", ""),
            }

    return closest

def get_lottecinema_schedule(cinema_id: str, detail_division_code: str):
    url = f"https://www.lottecinema.co.kr/NLCHS/Cinema/Detail?divisionCode=1&detailDivisionCode={detail_division_code}&cinemaID={cinema_id}"

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get(url)
    time.sleep(3)  # JS 렌더링 대기

    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    movie_blocks = soup.select("div.time_select_wrap.ty2.timeSelect")
    movies = []

    if not movie_blocks:
        print("상영 정보가 없습니다.")
        return movies

    for movie in movie_blocks:
        title_tag = movie.select_one("div.list_tit > p")
        title = title_tag.text.strip() if title_tag else "제목 없음"
        time_tags = movie.select("ul.list_time li dd.time strong")
        times = [t.text.strip() for t in time_tags]
        movies.append({
            "title": title,
            "times": times
        })

    return movies

if __name__ == "__main__":
    # 사용자 위치 예시 (가산디지털단지)
    user_lat = 37.4812
    user_lng = 126.8828

    nearest = find_nearest_lottecinema(user_lat, user_lng)

    if nearest:
        print(f"가장 가까운 롯데시네마: {nearest['name']} ({nearest['distance']} km)")
        cinema_id, detail_code = None, None
        for key in cinema_map:
            if key in nearest['name']:
                cinema_id, detail_code = cinema_map[key]
                break

        if cinema_id and detail_code:
            print(f"매핑된 cinemaID: {cinema_id}, detailDivisionCode: {detail_code}")
            schedule = get_lottecinema_schedule(cinema_id, detail_code)
            for movie in schedule:
                print(f"🎬 {movie['title']}")
                print(f"🕒 상영 시간: {', '.join(movie['times'])}")
                print("------")
        else:
            print("해당 지점에 대한 cinemaID 매핑이 없습니다.")
    else:
        print("근처에 롯데시네마가 없습니다.")


가장 가까운 롯데시네마: 롯데시네마 가산디지털 (0.66 km)
매핑된 cinemaID: 1013, detailDivisionCode: 1
🎬 전지적 독자 시점
🕒 상영 시간: 13:00, 14:05, 15:30, 16:40, 18:00, 19:10, 20:30, 21:40, 23:00
------
🎬 F1 더 무비
🕒 상영 시간: 14:15, 19:25, 22:30
------
🎬 킹 오브 킹스
🕒 상영 시간: 12:55, 16:45, 19:00
------
🎬 판타스틱 4: 새로운 출발
🕒 상영 시간: 13:30, 16:00, 18:25, 20:55, 23:25
------
🎬 명탐정 코난: 척안의 잔상
🕒 상영 시간: 14:40, 19:45, 22:05
------
🎬 베베핀 극장판: 사라진 베베핀과 핑크퐁 대모험
🕒 상영 시간: 15:10
------
🎬 극장판 도라에몽: 진구의 그림이야기
🕒 상영 시간: 12:25
------
🎬 쥬라기 월드: 새로운 시작
🕒 상영 시간: 17:00
------
🎬 노이즈
🕒 상영 시간: 17:20, 21:20
------


In [3]:
import requests
from geopy.distance import geodesic
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

# 카카오맵 API 키
KAKAO_API_KEY = "b3e4c845d73f26ab456c89a38561b720"

# cinemaID 매핑
cinema_map = {
    "가산": ("1013", "1"),
    "건대": ("1004", "1"),
    "동탄": ("3048", "2"),
    "서청주": ("4004", "3"),
    # 필요 시 추가
}

def search_nearest_lotte_by_name(query: str):
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {
        "query": f"롯데시네마 {query}",
        "radius": 20000,
        "size": 3
    }
    res = requests.get("https://dapi.kakao.com/v2/local/search/keyword.json", params=params, headers=headers)
    places = res.json().get("documents", [])

    if not places:
        return None

    # 거리 계산 생략 (query로 직접 검색했기 때문)
    place = places[0]
    return {
        "name": place["place_name"],
        "lat": float(place["y"]),
        "lng": float(place["x"]),
        "address": place.get("road_address_name", "")
    }

def get_lottecinema_schedule(cinema_id: str, detail_division_code: str):
    url = f"https://www.lottecinema.co.kr/NLCHS/Cinema/Detail?divisionCode=1&detailDivisionCode={detail_division_code}&cinemaID={cinema_id}"

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get(url)
    time.sleep(3)  # JS 렌더링 대기

    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    movie_blocks = soup.select("div.time_select_wrap.ty2.timeSelect")
    movies = []

    for movie in movie_blocks:
        title_tag = movie.select_one("div.list_tit > p")
        title = title_tag.text.strip() if title_tag else "제목 없음"
        time_tags = movie.select("ul.list_time li dd.time strong")
        times = [t.text.strip() for t in time_tags]
        if times:
            movies.append({
                "title": title,
                "times": times
            })

    return movies

def main(message: str):
    location_name = message.strip()

    if location_name in cinema_map:
        cinema_id, detail_code = cinema_map[location_name]
        result_name = f"롯데시네마 {location_name}"
    else:
        print(f"⚠️ 매핑 정보 없음: 카카오맵으로 검색 중...")
        place = search_nearest_lotte_by_name(location_name)
        if not place:
            print("❌ 해당 위치에 가까운 롯데시네마를 찾을 수 없습니다.")
            return

        result_name = place["name"]
        # 매핑 시도
        matched = None
        for key in cinema_map:
            if key in result_name:
                matched = key
                break

        if not matched:
            print("❌ 자동 매핑 실패: cinemaID 정보를 찾을 수 없습니다.")
            return

        cinema_id, detail_code = cinema_map[matched]

    # 스케줄 가져오기
    movies = get_lottecinema_schedule(cinema_id, detail_code)
    if not movies:
        print(f"🎥 {result_name}")
        print("현재 상영 중인 영화가 없습니다.")
        return

    print(f"🎥 {result_name}")
    for movie in movies:
        print(f"🎬 {movie['title']}")
        print(f"🕒 상영 시간: {', '.join(movie['times'])}")
        print("------")

# 예시 호출
if __name__ == "__main__":
    user_message = input("영화관 위치를 입력하세요 (예: 건대, 가산): ")
    main(user_message)


⚠️ 매핑 정보 없음: 카카오맵으로 검색 중...
❌ 해당 위치에 가까운 롯데시네마를 찾을 수 없습니다.


In [12]:
import requests
import time
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from geopy.distance import geodesic
from fuzzywuzzy import process

# --- 설정 영역 ---
KAKAO_API_KEY = "b3e4c845d73f26ab456c89a38561b720"

cinema_map = {
'가산디지털': ('1013', '1'), 
'가양': ('9094', '1'), 
'강동': ('9010', '1'), 
'건대입구': ('1004', '1'), 
'김포공항': ('1009', '1'), 
'노원': ('1003', '1'), 
'도곡': ('1023', '1'), 
'독산': ('1017', '1'), 
'서울대입구': ('1012', '1'), 
'수락산': ('9099', '1'), 
'수유': ('9104', '1'), 
'신대방(구로디지털역)': ('1024', '1'), 
'신도림': ('1015', '1'), 
'신림': ('1007', '1'), 
'에비뉴엘(명동)': ('1001', '1'), 
'영등포': ('1002', '1'), 
'용산': ('1014', '1'), 
'월드타워': ('1016', '1'), 
'은평(롯데몰)': ('1021', '1'), 
'중랑': ('9083', '1'), 
'청량리': ('1008', '1'), 
'합정': ('1010', '1'), 
'홍대입구': ('1005', '1'), 
'광교': ('3030', '2'), 
'광명(광명사거리)': ('3027', '2'), 
'광명아울렛': ('3025', '2'), 
'구리아울렛': ('3026', '2'), 
'동탄': ('3048', '2'), 
'라페스타': ('9095', '2'), 
'마석': ('3021', '2'), 
'별내': ('3046', '2'), 
'병점': ('3017', '2'), 
'부천(신중동역)': ('3011', '2'), 
'부평': ('3003', '2'), 
'부평갈산': ('3050', '2'), 
'부평역사': ('3008', '2'), 
'북수원(천천동)': ('3045', '2'), 
'산본피트인': ('3031', '2'), 
'서수원': ('3043', '2'), 
'성남중앙(신흥역)': ('3041', '2'), 
'센트럴락': ('3012', '2'), 
'송탄': ('3029', '2'), 
'수원(수원역)': ('3024', '2'), 
'수지': ('3044', '2'), 
'시화(정왕역)': ('9088', '2'), 
'시흥장현': ('3049', '2'), 
'안산': ('3004', '2'), 
'안산고잔': ('3028', '2'), 
'안성': ('9106', '2'), 
'안양(안양역)': ('3007', '2'), 
'안양일번가': ('3032', '2'), 
'용인기흥': ('3039', '2'), 
'용인역북': ('3040', '2'), 
'위례': ('3037', '2'), 
'의정부민락': ('3033', '2'), 
'인덕원': ('9100', '2'), 
'인천아시아드': ('3035', '2'), 
'인천터미널': ('3038', '2'), 
'진접': ('3010', '2'), 
'파주롯데아울렛': ('9113', '2'), 
'파주운정': ('3034', '2'), 
'판교(창조경제밸리)': ('3047', '2'), 
'평촌(범계역)': ('3018', '2'), 
'하남미사': ('9111', '2'), 
'향남': ('3036', '2'), 
'당진': ('9085', '3'), 
'대전(백화점)': ('4002', '3'), 
'대전관저': ('4009', '3'), 
'대전센트럴': ('4008', '3'), 
'서산': ('9044', '3'), 
'서청주(아울렛)': ('4004', '3'), 
'아산터미널': ('4005', '3'), 
'오송': ('9119', '3'), 
'천안불당': ('9101', '3'), 
'천안청당': ('9112', '3'), 
'청주용암': ('4007', '3'), 
'충주(모다아울렛)': ('9078', '3'), 
'광주(백화점)': ('6001', '4'), 
'광주광산': ('9065', '4'), 
'광주첨단': ('9117', '4'), 
'군산나운': ('6007', '4'), 
'군산몰': ('6009', '4'), 
'수완(아울렛)': ('6004', '4'), 
'익산모현': ('9070', '4'), 
'전주(백화점)': ('6002', '4'), 
'전주송천': ('9102', '4'), 
'전주평화': ('6006', '4'), 
'충장로': ('9047', '4'), 
'경주황성': ('9120', '5'), 
'구미공단': ('5013', '5'), 
'대구광장': ('5012', '5'), 
'대구율하': ('5006', '5'), 
'대구현풍': ('9121', '5'), 
'동성로': ('5005', '5'), 
'상인': ('5016', '5'), 
'상주': ('9080', '5'), 
'성서': ('5004', '5'), 
'영주': ('9064', '5'), 
'영천': ('9098', '5'), 
'포항': ('9097', '5'), 
'프리미엄구미센트럴': ('9067', '5'), 
'프리미엄안동': ('9074', '5'), 
'프리미엄칠곡': ('9057', '5'), 
'거창': ('9122', '101'), 
'광복': ('2009', '101'), 
'김해부원': ('5015', '101'), 
'김해아울렛(장유)': ('5011', '101'), 
'동래': ('2007', '101'), 
'동부산아울렛': ('2010', '101'), 
'마산(합성동)': ('9042', '101'), 
'부산명지': ('9092', '101'), 
'부산본점': ('2004', '101'), 
'부산장림': ('9115', '101'), 
'사천': ('9084', '101'),
'센텀시티': ('2006', '101'), 
'양산물금': ('9103', '101'), 
'엠비씨네(진주)': ('9105', '101'), 
'오투(부산대)': ('2011', '101'), 
'울산(백화점)': ('5001', '101'), 
'울산성남': ('9116', '101'), 
'진주혁신(롯데몰)': ('5017', '101'), 
'창원': ('5002', '101'),  
'통영': ('9036', '101'), 
'프리미엄경남대': ('9072', '101'), 
'프리미엄해운대(장산역)': ('9059', '101'), 
'강릉': ('9118', '6'), 
'남원주': ('9108', '6'), 
'동해': ('7002', '6'), 
'원주무실': ('7003', '6'), 
'춘천': ('9081', '6'), 
'서귀포': ('9013', '7'), 
'제주연동': ('6010', '7')
}

# --- 1. 지점 검색 (없으면 카카오맵 API 보완) ---
def search_lotte_cinema_place(query):
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": f"롯데시네마 {query}", "size": 1}
    res = requests.get("https://dapi.kakao.com/v2/local/search/keyword.json", params=params, headers=headers)
    data = res.json().get("documents", [])
    if data:
        place = data[0]
        return {
            "name": place["place_name"],
            "lat": float(place["y"]),
            "lng": float(place["x"]),
            "address": place.get("road_address_name", "")
        }
    return None

# --- 2. 상영 정보 크롤링 ---
def get_lottecinema_schedule(cinema_id, detail_code):
    url = f"https://www.lottecinema.co.kr/NLCHS/Cinema/Detail?divisionCode=1&detailDivisionCode={detail_code}&cinemaID={cinema_id}"
    options = Options()
    options.add_argument('--headless')
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get(url)
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    movie_blocks = soup.select("div.time_select_wrap.ty2.timeSelect")
    movie_info = []
    for block in movie_blocks:
        title_tag = block.select_one("div.list_tit > p")
        time_tags = block.select("ul.list_time li dd.time strong")
        if title_tag and time_tags:
            title = title_tag.text.strip()
            times = [t.text.strip() for t in time_tags]
            movie_info.append({"title": title, "times": times})
    return movie_info

# --- 3. 유사 영화 제목 찾기 ---
def find_closest_movie(movie_title, movie_list):
    titles = [m["title"] for m in movie_list]
    match, score = process.extractOne(movie_title, titles)
    if score >= 70:
        for m in movie_list:
            if m["title"] == match:
                return m
    return None

# --- 4. 입력 분석 및 처리 ---
def handle_user_message(message):
    tokens = message.strip().split()
    if not tokens:
        print("입력값이 없습니다.")
        return

    # 지점 + (선택적) 영화 제목 분리
    location = tokens[0]
    movie_query = " ".join(tokens[1:]) if len(tokens) > 1 else None

    # 지점 매핑 확인
    if location in cinema_map:
        cinema_id, detail_code = cinema_map[location]
        result_name = f"롯데시네마 {location}"
    else:
        # 카카오맵 API로 보완
        place = search_lotte_cinema_place(location)
        if not place:
            print("❌ 해당 위치의 롯데시네마를 찾을 수 없습니다.")
            return
        matched_key = next((key for key in cinema_map if key in place["name"]), None)
        if not matched_key:
            print(f"입력한 '{location}' 지점은 없습니다. 가장 가까운 롯데시네마는 '{place['name']}' 입니다.")
            return
        cinema_id, detail_code = cinema_map[matched_key]
        result_name = f"롯데시네마 {matched_key} (가장 가까운 지점)"

    # 영화 정보 가져오기
    movie_list = get_lottecinema_schedule(cinema_id, detail_code)
    if not movie_list:
        print(f"{result_name}에 현재 상영 정보가 없습니다.")
        return

    # 영화 제목 입력 여부 확인
    if movie_query:
        matched_movie = find_closest_movie(movie_query, movie_list)
        if matched_movie:
            print(f"🎥 {result_name}")
            print(f"🎬 {matched_movie['title']}")
            print(f"🕒 상영 시간: {', '.join(matched_movie['times'])}")
        else:
            print(f"'{movie_query}'에 해당하는 영화가 {result_name}에서 검색되지 않았습니다.")
    else:
        # 전체 상영작 출력
        print(f"🎥 {result_name} - 현재 상영작:")
        for m in movie_list:
            print(f"🎬 {m['title']}")
            print(f"🕒 {', '.join(m['times'])}")
            print("------")

# --- 예시 실행 ---
if __name__ == "__main__":
    user_input = input("위치 또는 '지점 영화제목'을 입력하세요: ")
    handle_user_message(user_input)


🎥 롯데시네마 가산디지털 (가장 가까운 지점)
🎬 전지적 독자 시점
🕒 상영 시간: 16:40, 18:00, 19:10, 20:30, 21:40, 23:00


In [ ]:
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# 1. 크롬 드라이버 생성 (헤드리스 모드)
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=options)

# 2. 대상 URL 리스트 (cinemaID, divisionCode)
cinema_list = [
    (1013, 1), (9094, 1), (9010, 1), (1004, 1), (1009, 1),
    (1003, 1), (1023, 1), (1017, 1), (1012, 1), (9099, 1),
    (9104, 1), (1024, 1), (1015, 1), (1007, 1), (1001, 1),
    (1002, 1), (1014, 1), (1016, 1), (1021, 1), (9083, 1),
    (1008, 1), (1010, 1), (1005, 1), (3030, 2), (3027, 2), 
    (3025, 2), (3026, 2), (3048, 2), (9095, 2), (3021, 2), 
    (3046, 2), (3017, 2), (3011, 2), (3003, 2), (3050, 2), 
    (3008, 2), (3045, 2), (3031, 2), (3043, 2), (3041, 2), 
    (3012, 2), (3029, 2), (3024, 2), (3044, 2), (9088, 2), 
    (3049, 2), (3004, 2), (3028, 2), (9106, 2), (3007, 2), 
    (3032, 2), (3039, 2), (3040, 2), (3037, 2), (3033, 2),
    (9100, 2), (3035, 2), (3038, 2), (3010, 2), (9113, 2), 
    (3034, 2), (3047, 2), (3018, 2), (9111, 2), (3036, 2), 
    (9085, 3), (4002, 3), (4009, 3), (4008, 3), (9044, 3), 
    (4004, 3), (4005, 3), (9119, 3), (9101, 3), (9112, 3), 
    (4007, 3), (9078, 3), (6001, 4), (9065, 4), (9117, 4), 
    (6007, 4), (6009, 4), (6004, 4), (9070, 4), (6002, 4), 
    (9102, 4), (6006, 4), (9047, 4), (9120, 5), (5013, 5), 
    (5012, 5), (5006, 5), (9121, 5), (5005, 5), (5016, 5), 
    (9080, 5), (5004, 5), (9064, 5), (9098, 5), (9097, 5), 
    (9067, 5), (9074, 5), (9057, 5), 
    (9122, 101), (2009, 101), (5015, 101), (5011, 101), (2007, 101), (2010, 101),
    (9042, 101), (9092, 101), (2004, 101), (9115, 101), (9084, 101), (2008, 101),
    (2006, 101), (9103, 101), (9105, 101), (2011, 101), (5001, 101), (9116, 101),
    (5017, 101), (5002, 101), (5002, 101), (9036, 101), (9072, 101), (9059, 101),
    (9118, 6), (9108, 6), (7002, 6), (7003, 6), (9081, 6), (9013, 7), (6010, 7)
]

result_dict = {}

for cid, div in cinema_list:
    url = f"https://www.lottecinema.co.kr/NLCHS/Cinema/Detail?divisionCode=1&detailDivisionCode={div}&cinemaID={cid}"
    driver.get(url)

    # 3. 충분히 대기 (10~20초 가능, 상황에 맞게 조절)
    time.sleep(15)

    html = driver.page_source

    # 4. 정규식으로 <h3 class="tit">지점명</h3> 텍스트 추출
    match = re.search(r'<h3 class="tit">\s*([^<]+?)\s*</h3>', html)
    if match:
        branch_name = match.group(1).strip()
        print(f"'{branch_name}': ('{cid}', '{div}'), ")
        result_dict[branch_name] = (str(cid), str(div))
    else:
        print(f"[⚠️] 못찾음: cinemaID={cid}, divisionCode={div}")

driver.quit()

# 5. 결과 출력
print("\n최종 딕셔너리:")
for k, v in result_dict.items():
    print(f'"{k}": {v},')


'센텀시티': ('2006', '101'), 
'양산물금': ('9103', '101'), 
'엠비씨네(진주)': ('9105', '101'), 
'오투(부산대)': ('2011', '101'), 
'울산(백화점)': ('5001', '101'), 
'울산성남': ('9116', '101'), 
'진주혁신(롯데몰)': ('5017', '101'), 
'창원': ('5002', '101'), 
'창원': ('5002', '101'), 
'통영': ('9036', '101'), 
'프리미엄경남대': ('9072', '101'), 
'프리미엄해운대(장산역)': ('9059', '101'), 
'강릉': ('9118', '6'), 
'남원주': ('9108', '6'), 
'동해': ('7002', '6'), 
'원주무실': ('7003', '6'), 
'춘천': ('9081', '6'), 
'서귀포': ('9013', '7'), 
'제주연동': ('6010', '7'), 

최종 딕셔너리:
"센텀시티": ('2006', '101'),
"양산물금": ('9103', '101'),
"엠비씨네(진주)": ('9105', '101'),
"오투(부산대)": ('2011', '101'),
"울산(백화점)": ('5001', '101'),
"울산성남": ('9116', '101'),
"진주혁신(롯데몰)": ('5017', '101'),
"창원": ('5002', '101'),
"통영": ('9036', '101'),
"프리미엄경남대": ('9072', '101'),
"프리미엄해운대(장산역)": ('9059', '101'),
"강릉": ('9118', '6'),
"남원주": ('9108', '6'),
"동해": ('7002', '6'),
"원주무실": ('7003', '6'),
"춘천": ('9081', '6'),
"서귀포": ('9013', '7'),
"제주연동": ('6010', '7'),
